In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from scipy.optimize import minimize_scalar
from scipy.optimize import minimize
np.set_printoptions(legacy='1.21')

In [2]:
#Minimizing a function with many variables and multiple constraints
# 1. Objective Function
def f(x):
    return 2*x[0]**2 - 5*x[0] + 3*x[1]**2

constraints = [
            {'type': 'ineq', 'fun': lambda x: 1-x[0] - x[1]}, 
        {'type': 'ineq', 'fun': lambda x: 0.75 - x[0]}
        ]

initial_guess = [0, 0]

result = minimize(f, initial_guess, method='SLSQP', constraints=constraints)
result
#method='SLSQP' stands for Sequential Least Squares Programming 
#It is an optimization algorithm used to find the minimum (or maximum) of a function, 
#particularly when you have complex constraints.

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -2.624999999999967
       x: [ 7.500e-01 -4.470e-08]
     nit: 2
     jac: [-2.000e+00 -2.384e-07]
    nfev: 6
    njev: 2

In [3]:
#Portfolio two assets
r0=0.06
r1=0.09
def portreturn(x):
        return -r0*x[0]- r1*x[1]

constraints = [
            {'type': 'ineq', 'fun': lambda x: 1-x[0] },  # Inquality: 1-x[0]  >= 0
            {'type': 'ineq', 'fun': lambda x: 1-x[1]}, # Inequality: 1 - x[1] >= 0            
            {'type': 'ineq', 'fun': lambda x: x[0]}, # Inequality: x[0] >= 0
            {'type': 'ineq', 'fun': lambda x: x[1]}, # Inequality: x[1] >= 0
            {'type': 'ineq', 'fun': lambda x: 1-x[0]-x[1]}, # Inequality: 1-x[0]-x[1] >= 0
        ]

initial_guess = [0.5, 0.5]

result = minimize(portreturn, initial_guess, method='SLSQP', constraints=constraints)
result

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -0.09
       x: [-7.078e-16  1.000e+00]
     nit: 5
     jac: [-6.000e-02 -9.000e-02]
    nfev: 15
    njev: 5

In [4]:
#Portfolio two assets and constraint on volatility
r0=0.06
r1=0.09
s0=0.1
s1=0.2
rho01=0.3

def portfoliovolatility(x):
    a=np.sqrt((x[0]*s0)**2+(x[1]*s1)**2+2*rho01*x[0]*x[1]*s1*s0)
    return a

def portreturn(x):
        return -r0*x[0]- r1*x[1]

constraints = [
            {'type': 'ineq', 'fun': lambda x: 1-x[0] },  # Inquality: 1-x[0]  >= 0
            {'type': 'ineq', 'fun': lambda x: 1-x[1]}, # Inequality: 1 - x[1] >= 0            
            {'type': 'ineq', 'fun': lambda x: x[0]}, # Inequality: x[0] >= 0
            {'type': 'ineq', 'fun': lambda x: x[1]}, # Inequality: x[1] >= 0
            {'type': 'eq', 'fun': lambda x: 1-x[0]-x[1]}, # Inequality: 1-x[0]-x[1] >= 0
            {'type': 'ineq', 'fun': lambda x: 0.12-portfoliovolatility(x)}, # Volatility lower than 12%

        ]

initial_guess = [0.5, 0.5]

result = minimize(portreturn, initial_guess, method='SLSQP', constraints=constraints)
result

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -0.07384357181453928
       x: [ 5.385e-01  4.615e-01]
     nit: 3
     jac: [-6.000e-02 -9.000e-02]
    nfev: 9
    njev: 3

In [5]:
xstar=[0.5385,0.4615]
print(portfoliovolatility(xstar))
print(-portreturn(xstar))

0.12000552278957832
0.073845


In [6]:
#A more general approach with an arbitrary number of assets.
#Compute portfolio volatility given weights, standard deviations, and correlation matrix.
#weights : array-like, shape (n_assets,). Portfolio weights (should sum to 1).
#std_devs : array-like, shape (n_assets,). Standard deviations of individual assets.
#corr_matrix : array-like, shape (n_assets, n_assets). Correlation matrix between assets.

def portfolio_volatility(weights, std_devs, corr_matrix):
    # Convert to numpy arrays
    w = np.asarray(weights)
    s = np.asarray(std_devs)
    C = np.asarray(corr_matrix)
    
    # Build covariance matrix: Σ = D * C * D
    # The symbol @ performs matrix multiplication in Python.
    D = np.diag(s)
    Sigma = D @ C @ D
    
    # Portfolio variance and volatility
    var_p = w.T @ Sigma @ w
    vol_p = np.sqrt(var_p)
    
    return vol_p


In [7]:
#The construction of the covariance matrix
s=[0.1,0.2]        #vector of standard deviation
D=np.diag(s)       #creates a matrix with standard deviations on the diagonal
D

array([[0.1, 0. ],
       [0. , 0.2]])

In [8]:
C= np.array([                  #Correlation matrix
    [1.0, 0.2],
    [0.2, 1.0]
])

A = D @ C @ D                 #Generates the covariance matrix through matrix multiplication

A

array([[0.01 , 0.004],
       [0.004, 0.04 ]])

In [9]:
#The variance of the portfolio is obtained applying the vectors of weights 
#to the variance/covariance matrix
w=np.array([0.5,0.5])
sigmap = w.T@A@ w
print(sigmap)
print(np.sqrt(sigmap))

0.014500000000000002
0.12041594578792296


In [10]:
#An example with 3 assets
weights = np.array([0, 0, 1])
std_devs = np.array([0.10, 0.15, 0.20])
corr_matrix = np.array([
    [1.0, 0.2, 0.1],
    [0.2, 1.0, 0.4],
    [0.1, 0.4, 1.0]
])

portfolio_volatility(weights,std_devs,corr_matrix)

0.2

In [11]:
#An example with 4 assets and a equally weighted portfolio
n=4
w=[1/n for i in range(n)] 
w

[0.25, 0.25, 0.25, 0.25]

In [12]:
#Create a vector of standard deviations
s=[(i+1)/n for i in range(n)]
s

[0.25, 0.5, 0.75, 1.0]

In [13]:
#Assume all assets are independent, so correlation zero off diagonal
D=np.eye(n)
D

array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])

In [14]:
#The variance matrix
A=np.diag(s)
A@D@A

array([[0.0625, 0.    , 0.    , 0.    ],
       [0.    , 0.25  , 0.    , 0.    ],
       [0.    , 0.    , 0.5625, 0.    ],
       [0.    , 0.    , 0.    , 1.    ]])

In [15]:
portfolio_volatility(w,s,D)

0.3423265984407288

In [16]:
#We now have learnt how to compute the volatility of a portfolio in general settings.
#Consider expected return maximization
n=4
r=[0.02*(i+1) for i in range(n)]
r

[0.02, 0.04, 0.06, 0.08]

In [17]:
#remember that w is the equally weighted portfolio
er=sum(w[i]*r[i] for i in range(n))
er

0.05